# 03 · Cluster Validation & Stability
### Lifestyle Archetypes and Problematic Internet Use — Pipeline Notebook 3 of 5

**This notebook covers:**
8. Cluster validation (silhouette, Davies-Bouldin, Calinski-Harabasz)
9. Stability analysis (subsampling + noise sensitivity)
10. Selection of the final clustering solution to carry forward

**Loads:** `01_data_preparation.pkl`, `02_dimensionality_reduction_clustering.pkl`
**Produces:** `03_validation_stability.pkl`, consumed by `04_archetype_characterisation.ipynb`.

## Setup & Environment

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from sklearn.cluster import KMeans
import umap.umap_ as umap


### Load artifacts from notebooks 01-02

In [ ]:
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

with open(ARTIFACT_DIR / "01_data_preparation.pkl", "rb") as f:
    artifact_01 = pickle.load(f)

with open(ARTIFACT_DIR / "02_dimensionality_reduction_clustering.pkl", "rb") as f:
    artifact_02 = pickle.load(f)

model_df = artifact_01["model_df"]
CLUSTER_FEATURES = artifact_01["CLUSTER_FEATURES"]

X_scaled = artifact_02["X_scaled"]
embedding_5d = artifact_02["embedding_5d"]
UMAP_N_NEIGHBORS = artifact_02["UMAP_N_NEIGHBORS"]
UMAP_MIN_DIST = artifact_02["UMAP_MIN_DIST"]
best_k = artifact_02["best_k"]
labels_kmeans = artifact_02["labels_kmeans"]
labels_agg = artifact_02["labels_agg"]
labels_dbscan = artifact_02["labels_dbscan"]

print(f"Loaded embedding_5d {embedding_5d.shape}, best_k={best_k}")


## 8. Cluster Validation

Three complementary internal-validation metrics, each measuring something different:
- **Silhouette score** (-1 to 1, higher better): how well-separated and internally cohesive clusters are, on average, per point.
- **Davies-Bouldin index** (>=0, lower better): average similarity between each cluster and its most-similar other cluster - penalises clusters that are close together relative to their own spread.
- **Calinski-Harabasz index** (higher better): ratio of between-cluster to within-cluster dispersion - sensitive to the number of clusters, so it's read alongside the other two rather than alone.

We compare all three algorithms' chosen solutions on the same footing, and exclude DBSCAN's noise points (-1) from its own metric computation since those points are explicitly "no cluster", not a nominal cluster.

In [ ]:
def validation_row(name, X, labels):
    mask = labels != -1  # exclude DBSCAN noise points from validation of the clustered points
    n_clusters = len(set(labels[mask]))
    if n_clusters < 2:
        return {"method": name, "n_clusters": n_clusters, "silhouette": np.nan,
                "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
    return {
        "method": name,
        "n_clusters": n_clusters,
        "silhouette": silhouette_score(X[mask], labels[mask]),
        "davies_bouldin": davies_bouldin_score(X[mask], labels[mask]),
        "calinski_harabasz": calinski_harabasz_score(X[mask], labels[mask]),
    }

validation_table = pd.DataFrame([
    validation_row("K-means", embedding_5d, labels_kmeans),
    validation_row("Agglomerative", embedding_5d, labels_agg),
    validation_row("DBSCAN", embedding_5d, labels_dbscan),
]).round(3)

display(validation_table)


## 9. Stability Analysis

A clustering solution that looks clean on one run but collapses under mild perturbation is not a reliable finding. We test the K-means solution (our leading candidate, pending the validation results above) two ways:

1. **Subsampling stability:** repeatedly re-cluster random 80% subsamples and measure agreement (Adjusted Rand Index) with cluster labels on the shared, overlapping points.
2. **Noise sensitivity:** add small Gaussian noise to the standardised input features, re-embed and re-cluster, and measure agreement with the original labelling.

ARI ranges from ~0 (agreement no better than chance) to 1 (perfect agreement).

In [ ]:
N_TRIALS = 10
SUBSAMPLE_FRAC = 0.8

subsample_aris = []
rng = np.random.RandomState(42)
for trial in range(N_TRIALS):
    idx = rng.choice(len(embedding_5d), size=int(len(embedding_5d) * SUBSAMPLE_FRAC), replace=False)
    idx_sorted = np.sort(idx)
    km_sub = KMeans(n_clusters=best_k, n_init=10, random_state=trial).fit(embedding_5d[idx_sorted])
    ari = adjusted_rand_score(labels_kmeans[idx_sorted], km_sub.labels_)
    subsample_aris.append(ari)

print(f"Subsampling stability (k={best_k}, {N_TRIALS} trials, {SUBSAMPLE_FRAC:.0%} subsamples):")
print(f"  mean ARI = {np.mean(subsample_aris):.3f}  (std = {np.std(subsample_aris):.3f})")


In [ ]:
NOISE_SIGMA = 0.1  # in standardised-feature units

noise_aris = []
for trial in range(N_TRIALS):
    rs = np.random.RandomState(100 + trial)
    X_noisy = X_scaled.to_numpy() + rs.normal(0, NOISE_SIGMA, X_scaled.shape)
    reducer_noisy = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, n_components=5, random_state=trial
    )
    emb_noisy = reducer_noisy.fit_transform(X_noisy)
    km_noisy = KMeans(n_clusters=best_k, n_init=10, random_state=trial).fit(emb_noisy)
    ari = adjusted_rand_score(labels_kmeans, km_noisy.labels_)
    noise_aris.append(ari)

print(f"Noise sensitivity (sigma={NOISE_SIGMA}, {N_TRIALS} trials):")
print(f"  mean ARI = {np.mean(noise_aris):.3f}  (std = {np.std(noise_aris):.3f})")

print(
    "\nInterpretation guide: ARI > 0.6 suggests a reasonably stable solution; "
    "0.3-0.6 suggests moderate stability worth flagging as a limitation; "
    "< 0.3 suggests the solution should be treated as provisional/exploratory only."
)


## 10. Selecting the Final Solution

We adopt the K-means solution as the final archetype assignment by default (it is generally the strongest candidate on the validation table above, and its stability has just been quantified). To use a different method instead, change `FINAL_METHOD` below — `labels_agg` and `labels_dbscan` are both available.

In [ ]:
FINAL_METHOD = "kmeans"  # one of: "kmeans", "agglomerative", "dbscan"

_label_lookup = {
    "kmeans": labels_kmeans,
    "agglomerative": labels_agg,
    "dbscan": labels_dbscan,
}
FINAL_LABELS = _label_lookup[FINAL_METHOD]

print(f"Final method selected: {FINAL_METHOD}")
print("Cluster sizes:")
display(pd.Series(FINAL_LABELS).value_counts().sort_index())


## Save artifacts for downstream notebooks

In [ ]:
artifact = {
    "validation_table": validation_table,
    "subsample_aris": subsample_aris,
    "noise_aris": noise_aris,
    "N_TRIALS": N_TRIALS,
    "SUBSAMPLE_FRAC": SUBSAMPLE_FRAC,
    "NOISE_SIGMA": NOISE_SIGMA,
    "FINAL_METHOD": FINAL_METHOD,
    "FINAL_LABELS": FINAL_LABELS,
}

with open(ARTIFACT_DIR / "03_validation_stability.pkl", "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved artifact -> {ARTIFACT_DIR / '03_validation_stability.pkl'}")
